# Degradation-aware lung nodule malignancy screening — progress run (Colab GPU)

**Runtime → Change runtime type → GPU (T4 is enough; L4/A100 is ~2–3× faster).**

What this notebook does, top to bottom (≈ 2–2.5 h on a T4, mostly unattended):

| step | what | time (T4) |
|---|---|---|
| 1 | GPU check, clone repo, install | 1 min |
| 2 | unit + end-to-end smoke tests on synthetic data | 3 min |
| 3 | download LUNA25 annotations + nodule blocks (6.8 GB) + I3D weights | 10–20 min |
| 4 | data audit, patient-level split, crop cache | 15–25 min |
| 5 | train `baseline`, `aug`, `directional` (+ optional `symmetric`) | 60–90 min |
| 6 | evaluate: clean + 8 held-out degradations, referral at 80 % coverage | 10–15 min |
| 7 | figures + `headline.json` → zipped to Drive for the slides | 1 min |

Checkpoints and results are written to **Google Drive**, so a disconnect loses at most the current run.

In [ ]:
# 1a. GPU + Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/lungdeg'          # results survive disconnects here
ROOT = '/content/luna25'                          # fast local disk for data + cache
!mkdir -p {DRIVE}/runs {DRIVE}/raw {ROOT}
!ln -sfn {DRIVE}/runs {ROOT}/runs

In [ ]:
# 1b. Code. Public repo: just the URL. Private repo: add a GitHub token as a Colab secret named GH_TOKEN.
GITHUB_REPO = 'github.com/<OWNER>/lung-nodule-degradation-screening.git'   # <-- set once
import os
try:
    from google.colab import userdata
    tok = userdata.get('GH_TOKEN')
except Exception:
    tok = None
url = f'https://{tok}@{GITHUB_REPO}' if tok else f'https://{GITHUB_REPO}'
if not os.path.exists('/content/repo'):
    !git clone -q {url} /content/repo
else:
    !git -C /content/repo pull -q
!pip -q install -e /content/repo pytest
%cd /content/repo

## 2. Tests (fail fast before spending GPU time)

In [ ]:
!python -m pytest -q -m "not slow"
!python -m pytest -q -m slow -x     # trains 2 tiny models on synthetic crops, evaluates, builds report

## 3. Data download
Zip parts are kept on Drive after the first download (6.8 GB of the 15 GB free quota), so later sessions skip this step.

In [ ]:
!apt-get -qq install -y p7zip-full > /dev/null
!mkdir -p {ROOT}/raw && cp -n {DRIVE}/raw/* {ROOT}/raw/ 2>/dev/null; ls -la {ROOT}/raw
!python -m lungdeg.data.download --root {ROOT}
KEEP_RAW_ON_DRIVE = True
if KEEP_RAW_ON_DRIVE:
    !cp -n {ROOT}/raw/*.zip.00* {ROOT}/raw/*.csv {ROOT}/raw/model_rgb.pth {DRIVE}/raw/

## 4. Audit, patient-level split (70/15/15, dev → head_fit/calib/thresh), crop cache
The split CSV and audit JSON are copied to Drive — **these files go in the repo** (`splits/`) so every member uses the same split.

In [ ]:
!python -m lungdeg.prepare --root {ROOT} --seed 2025 --workers 2
!mkdir -p {DRIVE}/splits && cp {ROOT}/splits/* {DRIVE}/splits/
import json, pandas as pd
print(json.dumps(json.load(open(f'{ROOT}/splits/data_audit.json')), indent=1)[:3000])

## 5. Training
Same I3D, same hyper-parameters (official baseline: Adam 1e-4, wd 5e-4, batch 32, 10 epochs, ±20° rotation, 2.5 mm shift).
Only the training objective differs. Set `RUN_SYMMETRIC=False` if GPU time is short — it is the ablation, not needed for this week's comparison.

In [ ]:
import subprocess, sys
EPOCHS = 10
SEED = 2025
RUN_SYMMETRIC = True
methods = ['baseline', 'aug', 'directional'] + (['symmetric'] if RUN_SYMMETRIC else [])
for m in methods:
    out = f'{ROOT}/runs/{m}_s{SEED}'
    if os.path.exists(f'{out}/best.json'):
        print('already trained:', m); continue
    print(f'===== {m} =====', flush=True)
    !python -m lungdeg.train --root {ROOT} --method {m} --seed {SEED} --epochs {EPOCHS} --workers 2

## 6. Evaluation (no test label is used for any threshold, temperature or head)

In [ ]:
for m in methods:
    run = f'{ROOT}/runs/{m}_s{SEED}'
    if os.path.exists(f'{run}/test_referral.csv'):
        print('already evaluated:', m); continue
    !python -m lungdeg.evaluate --root {ROOT} --run {run} --tta 4 --n_boot 1000

## 7. Report → Drive (send `lungdeg_report.zip` back for the slides)

In [ ]:
!python -m lungdeg.report --root {ROOT} --out {DRIVE}/report
!cp {ROOT}/splits/data_audit.json {ROOT}/splits/split_seed2025_summary.json {DRIVE}/report/
!cd {DRIVE} && zip -qr lungdeg_report.zip report runs -x '*.pth' -x '*/preds/*'
from IPython.display import Image, display
for f in ['curves.png', 'auroc_by_condition.png', 'flr_at_coverage.png']:
    display(Image(f'{DRIVE}/report/{f}'))
print(open(f'{DRIVE}/report/headline.json').read())